In [1]:
from moabb.paradigms import MotorImagery
from moabb.datasets.utils import find_intersecting_channels
from brainbot_dataset import get_brainbot_dataset
from datasets16 import PhysionetMI16, Weibo2014_16

import moabb
import mne

moabb.set_log_level('INFO')
mne.set_log_level('INFO')

SUBJECTS = 6
MAX_TRIALS = 4

brainbot_dataset = get_brainbot_dataset()
brainbot_dataset.n_sessions = min(MAX_TRIALS, brainbot_dataset.n_sessions)
brainbot_dataset.subject_list = brainbot_dataset.subject_list[:SUBJECTS]
physionet16_dataset = PhysionetMI16()
physionet16_dataset.subject_list = physionet16_dataset.subject_list[:SUBJECTS]
# weibo2014_16_dataset = Weibo2014_16()
# weibo2014_16_dataset.subject_list = weibo2014_16_dataset.subject_list[:SUBJECTS]
# assert len(weibo2014_16_dataset.subject_list) == SUBJECTS
assert len(physionet16_dataset.subject_list) == SUBJECTS
assert len(brainbot_dataset.subject_list) == min(len(brainbot_dataset.subject_list), SUBJECTS)
assert brainbot_dataset.n_sessions == MAX_TRIALS


datasets = [brainbot_dataset, physionet16_dataset] # weibo2014_16_dataset
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

electrodes, datasets = find_intersecting_channels(datasets)
print("Datasets used:", [type(d).__name__ for d in datasets])
print("Used electrodes:", electrodes)

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)

Searching dataset: BrainBotDataset
Reading /home/mateusz/PROJEKTY/BCI2025-sandbox/brainbot_data/moabb-like/subject1_ses1_run1_20251206_185546_raw_epochs_splitted_segment=3.0-step=3.0-8-32Hz.epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2996.09 ms
        0 CTF compensation matrices available
Not setting metadata
50 matching events found
No baseline correction applied
0 projection items activated
Creating RawArray with float64 data, n_channels=17, n_times=38400
    Range : 0 ... 38399 =      0.000 ...   149.996 secs
Ready.
Reading /home/mateusz/PROJEKTY/BCI2025-sandbox/brainbot_data/moabb-like/subject1_ses1_run2_20251206_191128_raw_epochs_splitted_segment=3.0-step=3.0-8-32Hz.epo.fif ...
    Found the data of interest:
        t =       0.00 ...    2996.09 ms
        0 CTF compensation matrices available
Not setting metadata
50 matching events found
No baseline correction applied
0 projection items activated
Creating RawArray with float64 data, n_channels=17, 

/home/mateusz/PROJEKTY/BCI2025-sandbox/brainbot_dataset.py:55: RuntimeWarning: This filename (brainbot_data/moabb-like/subject1_ses1_run1_20251206_185546_raw_epochs_splitted_segment=3.0-step=3.0-8-32Hz.epo.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(path, preload=True)
/home/mateusz/PROJEKTY/BCI2025-sandbox/brainbot_dataset.py:55: RuntimeWarning: This filename (brainbot_data/moabb-like/subject1_ses1_run2_20251206_191128_raw_epochs_splitted_segment=3.0-step=3.0-8-32Hz.epo.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(path, preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['feet', 'hands', 'rest']
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['feet', 'hands', 'rest']
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['feet', 'hands', 'rest']
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Datasets used: ['BrainBotDataset', 'PhysionetMI16']
Used electrodes: ['C3', 'FC3', 'Pz', 'C4', 'Cz', 'CP3', 'CP4', 'CP2', 'C1', 'C2', 'FCz', 'CPz', 'CP1', 'FC2', 'FC1', 'FC4']


In [2]:
def print_results_summary(results_df):
    print("Results Summary:")
    summary = results_df.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count'])
    summary['mean'] = summary['mean'].round(3)
    summary['std'] = summary['std'].round(3)
    print(summary.to_string())
    print("=" * 50)

    print("\nDetailed Results by Subject and Dataset:")
    detailed = results_df.pivot_table(
        index=['dataset', 'subject', 'session'],
        columns='pipeline',
        values='score'
    )
    print(detailed.round(3).to_string())
    print("=" * 50)

### Test all builtin pipelines

In [13]:
from moabb import benchmark
import os

def run_moabb_benchmark(pipelines_dir, base_dir="./benchmarks", datasets_list=datasets, n_jobs=-1):
    pipelines_path = os.path.join(os.getcwd(), pipelines_dir)
    print(pipelines_path)

    cache_config = dict(
        use=True,
        save_raw=True,
        save_epochs=True,
        save_array=True,
        overwrite_raw=False,
        overwrite_epochs=False,
        overwrite_array=False,
    )

    print("Using cache dir:", mne.get_config('MNE_DATA'))

    return benchmark(
        pipelines=pipelines_path,
        evaluations=["WithinSession"],
        paradigms=["MotorImagery"],
        include_datasets=datasets_list,
        results=os.path.join(base_dir, f"results-{pipelines_dir}"),
        overwrite=True,
        plot=False,
        output=os.path.join(base_dir, f"output-{pipelines_dir}"),
        n_jobs=n_jobs,
        cache_config=cache_config
    )

results = run_moabb_benchmark("pipelines_MI")

/home/mateusz/PROJEKTY/BCI2025-sandbox/pipelines_MI
Using cache dir: /home/mateusz/mne_data


BrainBot-WithinSession:   0%|          | 0/3 [00:00<?, ?it/s]/home/mateusz/venvs/tf-gpu-wsl/lib/python3.12/site-packages/pyriemann/utils/mean.py:659: UserWarning: Convergence not reached
  warnings.warn("Convergence not reached")
/home/mateusz/venvs/tf-gpu-wsl/lib/python3.12/site-packages/pyriemann/utils/mean.py:659: UserWarning: Convergence not reached
  warnings.warn("Convergence not reached")
/home/mateusz/venvs/tf-gpu-wsl/lib/python3.12/site-packages/pyriemann/utils/mean.py:659: UserWarning: Convergence not reached
  warnings.warn("Convergence not reached")
/home/mateusz/venvs/tf-gpu-wsl/lib/python3.12/site-packages/pyriemann/utils/mean.py:659: UserWarning: Convergence not reached
  warnings.warn("Convergence not reached")
/home/mateusz/venvs/tf-gpu-wsl/lib/python3.12/site-packages/pyriemann/utils/mean.py:659: UserWarning: Convergence not reached
  warnings.warn("Convergence not reached")
/home/mateusz/venvs/tf-gpu-wsl/lib/python3.12/site-packages/pyriemann/utils/mean.py:659: UserW

                    dataset     evaluation                pipeline  avg score
0                  BrainBot  WithinSession               CSP + LDA   0.469788
1                  BrainBot  WithinSession                     MDM   0.482349
2                  BrainBot  WithinSession       DLCSPauto + shLDA   0.467695
3                  BrainBot  WithinSession          CSP + SVM Grid   0.478064
4                  BrainBot  WithinSession  Tangent Space SVM Grid   0.513058
5                  BrainBot  WithinSession        Tangent Space LR   0.530591
6                  BrainBot  WithinSession      TS ElasticNet Grid   0.521600
7                  BrainBot  WithinSession       AUG Tang SVM Grid   0.517091
8                  BrainBot  WithinSession                   FgMDM   0.514629
9   PhysionetMotorImagery16  WithinSession               CSP + LDA   0.453950
10  PhysionetMotorImagery16  WithinSession                     MDM   0.443305
11  PhysionetMotorImagery16  WithinSession       DLCSPauto + shL

In [14]:
print_results_summary(results)

Results Summary:
                                                 mean    std  count
pipeline               dataset                                     
AUG Tang SVM Grid      BrainBot                 0.517  0.084      6
                       PhysionetMotorImagery16  0.576  0.101      6
CSP + LDA              BrainBot                 0.470  0.077      6
                       PhysionetMotorImagery16  0.454  0.029      6
CSP + SVM Grid         BrainBot                 0.478  0.055      6
                       PhysionetMotorImagery16  0.493  0.022      6
DLCSPauto + shLDA      BrainBot                 0.468  0.073      6
                       PhysionetMotorImagery16  0.470  0.030      6
FgMDM                  BrainBot                 0.515  0.068      6
                       PhysionetMotorImagery16  0.511  0.142      6
MDM                    BrainBot                 0.482  0.070      6
                       PhysionetMotorImagery16  0.443  0.184      6
TS ElasticNet Grid     BrainBot

### Tensorflow pipelines have to be run separately due to OOM issues when running them in parallel

In [10]:
results = run_moabb_benchmark("pipelines_MI_tensorflow", n_jobs=1)
print_results_summary(results)

/home/mateusz/PROJEKTY/BCI2025-sandbox/pipelines_MI_tensorflow
Using cache dir: /home/mateusz/mne_data


I0000 00:00:1769895023.606262   66370 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13497 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9
BrainBot-WithinSession:   0%|          | 0/3 [00:00<?, ?it/s]/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 48 events (all good), 0 – 3 s (baseline off), ~4.5 MiB, data loaded,
 'rest': 9
 'left_hand': 10
 'right_hand': 10
 'hands': 10
 'feet': 9>
  warn(f"warnEpochs {epochs}")
/home/mateusz/PROJEKTY/moabb/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 48 events (all good), 0 – 3 s (baseline off), ~4.5 MiB, data loaded,
 'rest': 9
 'left_hand': 10
 'right_hand': 10
 'hands': 10
 'feet': 9>
  warn(f"warnEpochs {epochs}")
/home/mateusz/PROJEKTY/moabb/moabb/paradigms/base.py:350: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations 

float32


2026-01-31 22:34:38.819540: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


float32


...

float32


2026-01-31 23:36:15.387542: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-01-31 23:36:34.492783: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

float32


E0000 00:00:1769898998.512005   66370 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1499_1/dropout_1130_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-01-31 23:36:55.383406: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
Physione

                   dataset     evaluation              pipeline  avg score
0                 BrainBot  WithinSession  Keras_ShallowConvNet   0.402740
1                 BrainBot  WithinSession     Keras_DeepConvNet   0.225933
2                 BrainBot  WithinSession      Keras_EEGNet_8_2   0.295707
3                 BrainBot  WithinSession          Keras_EEGNeX   0.214455
4                 BrainBot  WithinSession        Keras_EEGITNet   0.213386
5  PhysionetMotorImagery16  WithinSession  Keras_ShallowConvNet   0.536387
6  PhysionetMotorImagery16  WithinSession     Keras_DeepConvNet   0.198908
7  PhysionetMotorImagery16  WithinSession      Keras_EEGNet_8_2   0.459608
8  PhysionetMotorImagery16  WithinSession          Keras_EEGNeX   0.402213
9  PhysionetMotorImagery16  WithinSession        Keras_EEGITNet   0.362241
Results Summary:
                                               mean    std  count
pipeline             dataset                                     
Keras_DeepConvNet    Brain